In [1]:
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

In [2]:
class YambdaDataset:
    INTERACTIONS = frozenset([
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ])

    def __init__(
        self,
        dataset_type: Literal["flat", "sequential"] = "flat",
        dataset_size: Literal["50m", "500m", "5b"] = "50m"
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(self, event_type: Literal[
        "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
    ]) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset("yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet")
        # Returns DatasetDict; extracting the only split
        assert isinstance(data, DatasetDict)
        return data["train"]

dataset = YambdaDataset("flat", "50m")
likes = dataset.interaction("likes")  # returns a HuggingFace Dataset


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

C:\Users\admin\.conda\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\admin\.cache\huggingface\hub\datasets--yandex--yambda. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\admin\.conda\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_h

flat/50m/likes.parquet: reconstructing file:   0%|          |  0.00B / 7.18MB            

flat/50m/likes.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

# Data Loading

Download the smallest version of Yambda dataset (50m interactions) from Hugging Face

In [3]:
yambda = YambdaDataset(dataset_type="flat", dataset_size="50m")
ds = yambda.interaction("multi_event")
print(f"Loaded {len(ds):,} records (multi_event)")

flat/50m/multi_event.parquet: reconstructing file:   0%|          |  0.00B /  384MB            

flat/50m/multi_event.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 47,790,449 records (multi_event)


In [4]:
# Load additional data: audio embeddings, album and artist mappings
print("Loading auto embedings...")
# audio_embeddings = yambda.audio_embeddings()
# df_audio = pl.from_arrow(audio_embeddings.data.table)
# print(f"Audio embeddings: {len(df_audio):,} tracks")
# print("\n=== Audio Embeddings (head) ===")
# df_audio.head(5)

Loading auto embedings...


In [6]:
print("Loading album-item mapping...")
album_mapping = yambda.album_item_mapping()
df_album = pl.from_arrow(album_mapping.data.table)
print(f"Album-item mapping: {len(df_album):,} rows")
print("\n=== Album-Item Mapping (head) ===")
df_album.head(5)

Loading album-item mapping...
Album-item mapping: 9,651,644 rows

=== Album-Item Mapping (head) ===


album_id,item_id
u32,u32
1,1491131
1,5109849
1,6735246
2,2859065
3,1859377


In [8]:
df = pl.from_arrow(ds.data.table)

# Print all unique values for is_organic and event_type
unique_is_organic = df["is_organic"].unique().sort().to_list()
unique_event_type = df["event_type"].unique().sort().to_list()

print("=== Unique values ===")
print(f"is_organic: {unique_is_organic}")
print(f"event_type: {unique_event_type}")

# Filter only listen events
df_listens = (
    df.filter(pl.col("event_type") == "listen").select(["uid", "timestamp", "item_id"])
)

print(f"Rows after filtering (event_type='listen'): {len(df_listens):,}")

=== Unique values ===
is_organic: [0, 1]
event_type: ['dislike', 'like', 'listen', 'undislike', 'unlike']
Rows after filtering (event_type='listen'): 46,467,212


In [9]:
df.head(10)

uid,timestamp,item_id,is_organic,played_ratio_pct,track_length_seconds,event_type
u32,u32,u32,u8,u16,u32,str
100,39420,8326270,0,100,170,"""listen"""
100,39420,1441281,0,100,105,"""listen"""
100,39625,286361,0,100,185,"""listen"""
100,40110,732449,0,100,240,"""listen"""
100,40360,3397170,0,46,130,"""listen"""
100,40380,7849270,0,100,205,"""listen"""
100,40640,1449307,0,100,205,"""listen"""
100,40785,6887232,0,100,145,"""listen"""
100,40885,2996197,0,100,95,"""listen"""


In [10]:
popularity = (
    df_listens.group_by("item_id")
    .agg(pl.len().alias("popularity"))
    .sort("popularity", descending=True)
    .with_row_index("rank")
)

print(popularity.head(10))

shape: (10, 3)
┌──────┬─────────┬────────────┐
│ rank ┆ item_id ┆ popularity │
│ ---  ┆ ---     ┆ ---        │
│ u32  ┆ u32     ┆ u32        │
╞══════╪═════════╪════════════╡
│ 0    ┆ 5862961 ┆ 41984      │
│ 1    ┆ 6901374 ┆ 40776      │
│ 2    ┆ 3542184 ┆ 39947      │
│ 3    ┆ 9378983 ┆ 38819      │
│ 4    ┆ 5635052 ┆ 38804      │
│ 5    ┆ 5463340 ┆ 38166      │
│ 6    ┆ 8213481 ┆ 37579      │
│ 7    ┆ 906358  ┆ 35870      │
│ 8    ┆ 3033749 ┆ 29977      │
│ 9    ┆ 2859641 ┆ 29602      │
└──────┴─────────┴────────────┘


In [11]:
n_users = df_listens["uid"].n_unique()
n_items = df_listens["item_id"].n_unique()
print(f"Number of unique users: {n_users:,}")
print(f"Number of unique items: {n_items:,}")

Number of unique users: 9,238
Number of unique items: 877,168
